# 04 — RAG Systems

Working notebook for Lewis et al. (2020) — Retrieval-Augmented Generation.

**Sections:**
1. The problem RAG solves
2. Chunking strategies
3. Building a vector index
4. Query routing
5. Retrieval and generation
6. RAG evaluation with RAGAS
7. Index freshness demonstration
8. Key observations

In [ ]:
# Install dependencies
%pip install anthropic chromadb sentence-transformers ragas datasets

In [ ]:
import anthropic
import chromadb
import json
import os
from sentence_transformers import SentenceTransformer

MODEL = "claude-sonnet-4-20250514"
client = anthropic.Anthropic()
embedder = SentenceTransformer("all-MiniLM-L6-v2")

print(f"Anthropic SDK version: {anthropic.__version__}")
print(f"Model: {MODEL}")
print(f"Embedder: {embedder._modules['0'].auto_model.config.name_or_path}")

---
## Section 1 — The Problem RAG Solves

We start with a mock FinMentor scenario. The naive approach sends the full portfolio snapshot for every question. We'll measure exactly how much of that context is irrelevant.

In [ ]:
import tiktoken

# Mock IBKR portfolio snapshot
portfolio = {
    "account_metadata": {
        "account_id": "U1234567",
        "account_type": "Individual",
        "base_currency": "USD",
        "nav": 287_450.00,
        "cash": 18_200.00,
        "margin_used": 0.0,
        "last_updated": "2026-05-19T09:30:00Z"
    },
    "positions": [
        {"ticker": "AAPL", "shares": 150, "avg_cost": 182.40, "current_price": 211.75, "sector": "Technology", "pnl_pct": 16.09},
        {"ticker": "MSFT", "shares": 80,  "avg_cost": 378.20, "current_price": 425.30, "sector": "Technology", "pnl_pct": 12.45},
        {"ticker": "NVDA", "shares": 50,  "avg_cost": 620.00, "current_price": 875.40, "sector": "Technology", "pnl_pct": 41.19},
        {"ticker": "GOOGL", "shares": 60, "avg_cost": 155.80, "current_price": 178.20, "sector": "Technology", "pnl_pct": 14.38},
        {"ticker": "JPM",  "shares": 100, "avg_cost": 198.50, "current_price": 221.40, "sector": "Financials", "pnl_pct": 11.54}
    ],
    "monthly_pnl": {
        "month": "2026-05",
        "realized_pnl": 4_820.50,
        "unrealized_pnl": 38_215.00,
        "total_pnl": 43_035.50,
        "best_performer": "NVDA",
        "worst_performer": "JPM"
    },
    "recent_transactions": [
        {"date": "2026-05-18", "ticker": "AAPL", "action": "BUY",  "shares": 10,  "price": 209.80},
        {"date": "2026-05-15", "ticker": "NVDA", "action": "SELL", "shares": 5,   "price": 882.10},
        {"date": "2026-05-12", "ticker": "MSFT", "action": "BUY",  "shares": 20,  "price": 418.50},
        {"date": "2026-05-10", "ticker": "GOOGL","action": "BUY",  "shares": 15,  "price": 172.30},
        {"date": "2026-05-08", "ticker": "JPM",  "action": "BUY",  "shares": 25,  "price": 215.70},
        {"date": "2026-05-05", "ticker": "AAPL", "action": "SELL", "shares": 5,   "price": 207.40},
        {"date": "2026-05-02", "ticker": "NVDA", "action": "BUY",  "shares": 10,  "price": 841.20},
        {"date": "2026-04-29", "ticker": "MSFT", "action": "SELL", "shares": 10,  "price": 411.80},
        {"date": "2026-04-25", "ticker": "GOOGL","action": "SELL", "shares": 5,   "price": 169.50},
        {"date": "2026-04-22", "ticker": "JPM",  "action": "BUY",  "shares": 30,  "price": 209.10}
    ]
}

snapshot_str = json.dumps(portfolio, indent=2)

enc = tiktoken.encoding_for_model("gpt-4")
total_tokens = len(enc.encode(snapshot_str))
print(f"Full snapshot: {total_tokens} tokens")
print(f"Characters: {len(snapshot_str)}")

# Two test questions
questions = [
    {
        "q": "What is my P&L this month?",
        "relevant_section": "monthly_pnl",
        "relevant_tokens": len(enc.encode(json.dumps(portfolio["monthly_pnl"], indent=2)))
    },
    {
        "q": "Should I worry about my tech concentration?",
        "relevant_section": "positions (sector field only)",
        "relevant_tokens": len(enc.encode(" ".join([p["ticker"] + " " + p["sector"] for p in portfolio["positions"]])))
    }
]

print("\n--- Naive approach: full snapshot sent for every question ---")
print(f"{'Question':<48} {'Tokens sent':>11} {'Relevant':>9} {'Irrelevant':>10} {'Wasted %':>9}")
print("-" * 92)
for item in questions:
    wasted = total_tokens - item["relevant_tokens"]
    wasted_pct = wasted / total_tokens * 100
    print(f"{item['q']:<48} {total_tokens:>11} {item['relevant_tokens']:>9} {wasted:>10} {wasted_pct:>8.1f}%")

This is FinMentor today. The relevant data for the P&L question is maybe 10% of what gets sent. The rest competes for attention and costs money. RAG replaces this with targeted retrieval — send only what the question needs.

---
## Section 2 — Chunking Strategies

Three strategies applied to the same IBKR portfolio data. Chunking quality determines retrieval quality more than embedding model choice.

In [ ]:
import textwrap

enc = tiktoken.encoding_for_model("gpt-4")

# ── 1. Fixed-size chunking ──────────────────────────────────────────────────
tokens = enc.encode(snapshot_str)
CHUNK_SIZE = 100
fixed_chunks = [
    enc.decode(tokens[i: i + CHUNK_SIZE])
    for i in range(0, len(tokens), CHUNK_SIZE)
]

print("=== Fixed-size chunking (100 tokens) ===")
print(f"Chunks produced: {len(fixed_chunks)}")
print("\nChunk 2 (notice mid-entry split):")
print(textwrap.fill(fixed_chunks[1], width=80))
print("\nChunk 3 (continuation — context lost without chunk 2):")
print(textwrap.fill(fixed_chunks[2][:300], width=80))

# ── 2. Semantic chunking ────────────────────────────────────────────────────
semantic_chunks = [
    ("positions",     json.dumps(portfolio["positions"], indent=2)),
    ("monthly_pnl",   json.dumps(portfolio["monthly_pnl"], indent=2)),
    ("transactions",  json.dumps(portfolio["recent_transactions"], indent=2)),
    ("account_meta",  json.dumps(portfolio["account_metadata"], indent=2)),
]

print("\n=== Semantic chunking (by logical boundary) ===")
for name, chunk in semantic_chunks:
    t = len(enc.encode(chunk))
    print(f"  {name:<20} {t:>4} tokens  — self-contained: True")

# ── 3. Hierarchical chunking ────────────────────────────────────────────────
summary_chunk = (
    "Portfolio summary: 5 positions across Technology (AAPL, MSFT, NVDA, GOOGL) "
    "and Financials (JPM). NAV $287,450. Monthly P&L $43,035. "
    "Best performer: NVDA +41%. Cash: $18,200."
)
detail_chunks = [
    (p["ticker"], json.dumps(p, indent=2))
    for p in portfolio["positions"]
]

print("\n=== Hierarchical chunking ===")
print(f"  summary              {len(enc.encode(summary_chunk)):>4} tokens  — entry point for all queries")
for ticker, chunk in detail_chunks:
    print(f"  detail:{ticker:<12} {len(enc.encode(chunk)):>4} tokens  — retrieved only when ticker is relevant")

# ── Comparison table ────────────────────────────────────────────────────────
avg_fixed = sum(len(enc.encode(c)) for c in fixed_chunks) / len(fixed_chunks)
avg_semantic = sum(len(enc.encode(c)) for _, c in semantic_chunks) / len(semantic_chunks)
avg_hier = (len(enc.encode(summary_chunk)) + sum(len(enc.encode(c)) for _, c in detail_chunks)) / (1 + len(detail_chunks))

print("\n" + "=" * 65)
print(f"{'Strategy':<16} {'Chunks':>6} {'Avg tokens':>11} {'Context preserved':>18}")
print("-" * 55)
print(f"{'Fixed':<16} {len(fixed_chunks):>6} {avg_fixed:>10.0f} {'Poor':>18}")
print(f"{'Semantic':<16} {len(semantic_chunks):>6} {avg_semantic:>10.0f} {'Good':>18}")
print(f"{'Hierarchical':<16} {1 + len(detail_chunks):>6} {avg_hier:>10.0f} {'Best':>18}")

For structured financial data like IBKR, semantic chunking is almost always right. Fixed-size loses meaning at chunk boundaries — a position entry split across two chunks means neither chunk is self-contained. Hierarchical is worth the complexity for large portfolios: retrieve the summary first, then drill into per-ticker detail only when the query needs it.

---
## Section 3 — Building a Vector Index

Two ChromaDB collections: `portfolio` (IBKR chunks) and `market_news` (analyst reports). Local embeddings via sentence-transformers — no API key needed for this step.

In [ ]:
# Mock analyst reports / news items
market_news = [
    {
        "id": "news_aapl_001",
        "ticker": "AAPL",
        "source": "Morgan Stanley",
        "date": "2026-05-17",
        "content": (
            "Apple maintained its Overweight rating from Morgan Stanley with a price target raised to $240. "
            "The analyst cited stronger-than-expected iPhone 17 pre-order data and improving Services segment "
            "margins driven by Apple Intelligence subscription uptake. Near-term risk remains macro sensitivity "
            "in China, where revenue declined 8% YoY in Q2 FY2026."
        )
    },
    {
        "id": "news_msft_001",
        "ticker": "MSFT",
        "source": "Goldman Sachs",
        "date": "2026-05-16",
        "content": (
            "Microsoft Azure growth re-accelerated to 31% YoY in Q3 FY2026, beating consensus by 300bps. "
            "Goldman Sachs raised its price target to $480 with a Buy rating, attributing the beat to "
            "Copilot for M365 enterprise seats exceeding 85 million. Operating margin expanded 120bps to 45.2%. "
            "The analyst flagged capex intensity as a watch item heading into FY2027."
        )
    },
    {
        "id": "news_nvda_001",
        "ticker": "NVDA",
        "source": "Barclays",
        "date": "2026-05-18",
        "content": (
            "Nvidia's Blackwell Ultra ramp is ahead of internal estimates according to Barclays channel checks, "
            "with GB300 shipments to hyperscalers already at 60% of full-quarter run rate in week three of May. "
            "The analyst reiterated Overweight with a $1,050 price target. Key risk: export control tightening "
            "on H20 sales to China could reduce FY2027 revenue by up to $4B."
        )
    },
    {
        "id": "news_googl_001",
        "ticker": "GOOGL",
        "source": "JPMorgan",
        "date": "2026-05-15",
        "content": (
            "Alphabet's Search revenue grew 13% YoY in Q1 2026 despite AI Overviews rollout concerns. "
            "JPMorgan maintained Overweight with a $210 price target, noting that AI Overviews are showing "
            "higher ad click-through rates than organic results. YouTube ad revenue +18% YoY was a standout. "
            "Waymo commercialization timeline risk remains a headline risk but not a near-term earnings driver."
        )
    },
    {
        "id": "news_jpm_001",
        "ticker": "JPM",
        "source": "Wells Fargo",
        "date": "2026-05-14",
        "content": (
            "JPMorgan Chase Q1 2026 results beat on NII but missed on investment banking fees. Wells Fargo "
            "maintained Equal Weight citing spread compression risk as the Fed signals two rate cuts in H2 2026. "
            "CET1 ratio of 15.3% is well above regulatory minimums. Buyback authorization of $30B provides "
            "downside support. Price target $230."
        )
    }
]

# Function to index documents into a ChromaDB collection
def index_documents(collection, chunks: list[dict], id_field: str, text_field: str):
    texts = [c[text_field] for c in chunks]
    embeddings = embedder.encode(texts).tolist()
    ids = [c[id_field] for c in chunks]
    metadatas = [{k: v for k, v in c.items() if k != text_field} for c in chunks]
    collection.add(
        ids=ids,
        embeddings=embeddings,
        documents=texts,
        metadatas=metadatas
    )
    return len(texts)

# Build ChromaDB client and collections
chroma = chromadb.Client()

portfolio_col = chroma.create_collection("portfolio")
news_col = chroma.create_collection("market_news")

# Prepare semantic portfolio chunks for indexing
portfolio_docs = [
    {"id": "pos_all",       "content": json.dumps(portfolio["positions"], indent=2),          "source": "ibkr", "type": "positions",     "ticker": "ALL"},
    {"id": "pnl_monthly",   "content": json.dumps(portfolio["monthly_pnl"], indent=2),        "source": "ibkr", "type": "pnl",          "ticker": "ALL"},
    {"id": "transactions",  "content": json.dumps(portfolio["recent_transactions"], indent=2),"source": "ibkr", "type": "transactions",  "ticker": "ALL"},
    {"id": "account_meta",  "content": json.dumps(portfolio["account_metadata"], indent=2),   "source": "ibkr", "type": "metadata",      "ticker": "ALL"},
]
# Add per-position detail chunks
for p in portfolio["positions"]:
    portfolio_docs.append({
        "id": f"pos_{p['ticker'].lower()}",
        "content": json.dumps(p, indent=2),
        "source": "ibkr",
        "type": "position_detail",
        "ticker": p["ticker"]
    })

n_portfolio = index_documents(portfolio_col, portfolio_docs, id_field="id", text_field="content")
n_news = index_documents(news_col, market_news, id_field="id", text_field="content")

sample_embedding = embedder.encode(["test"])[0]
print(f"Portfolio collection: {n_portfolio} documents indexed")
print(f"Market news collection: {n_news} documents indexed")
print(f"Embedding dimensions: {len(sample_embedding)}")

---
## Section 4 — Query Routing

One cheap Claude call before retrieval decides which index(es) to search. This prevents hitting every collection for every query.

In [ ]:
def classify_query(question: str) -> dict:
    """Route a question to the appropriate index(es) before retrieval."""
    response = client.messages.create(
        model=MODEL,
        max_tokens=256,
        system=(
            "Classify the user's financial question. Return JSON only — no markdown, no explanation.\n"
            "Schema: {\"intent\": string, \"indexes\": array, \"reasoning\": string}\n"
            "intent values: pnl | concentration | position | news | general\n"
            "indexes values (pick one or both): portfolio | market_news"
        ),
        messages=[{"role": "user", "content": question}]
    )
    raw = response.content[0].text.strip()
    # Strip markdown code fences if model returns them
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    return json.loads(raw)

test_questions = [
    "What is my P&L this month?",
    "Should I worry about my tech concentration?",
    "What did analysts say about NVDA this week?",
    "How is my portfolio performing overall?"
]

print("Query routing results:\n")
routing_results = {}
for q in test_questions:
    result = classify_query(q)
    routing_results[q] = result
    print(f"Q: {q}")
    print(f"   intent : {result['intent']}")
    print(f"   indexes: {result['indexes']}")
    print(f"   reason : {result['reasoning']}")
    print()

This is the query router. One cheap Claude call before retrieval prevents hitting every index for every query. In FinMentor this is what decides whether to call IBKR, hit the news index, or both — routing errors are silent correctness bugs because retrieval returns nothing useful and the model still generates an answer.

---
## Section 5 — Retrieval and Generation

Full pipeline: classify → route → retrieve → generate.

In [ ]:
COLLECTION_MAP = {
    "portfolio": portfolio_col,
    "market_news": news_col
}

def retrieve(question: str, collection_names: list[str], k: int = 3) -> list[dict]:
    """Embed the question, search specified collections, merge and re-rank."""
    query_embedding = embedder.encode([question])[0].tolist()
    all_results = []
    for name in collection_names:
        col = COLLECTION_MAP.get(name)
        if col is None:
            continue
        results = col.query(
            query_embeddings=[query_embedding],
            n_results=min(k, col.count())
        )
        for doc, dist, meta in zip(
            results["documents"][0],
            results["distances"][0],
            results["metadatas"][0]
        ):
            # ChromaDB returns L2 distance; convert to a 0-1 similarity score
            similarity = 1 / (1 + dist)
            all_results.append({"collection": name, "content": doc, "score": similarity, "metadata": meta})
    # Re-rank across collections by score descending
    all_results.sort(key=lambda x: x["score"], reverse=True)
    return all_results[:k]


def generate_answer(question: str, chunks: list[dict]) -> str:
    """Format retrieved chunks into context and call Claude."""
    context_block = "\n\n".join(
        f"[Source: {c['collection']} | score: {c['score']:.3f}]\n{c['content']}"
        for c in chunks
    )
    response = client.messages.create(
        model=MODEL,
        max_tokens=512,
        system=(
            "You are a financial assistant. Answer ONLY using the provided context. "
            "If the context is insufficient, say so clearly. "
            "Never use your training data to fill gaps."
        ),
        messages=[{
            "role": "user",
            "content": f"Context:\n{context_block}\n\nQuestion: {question}"
        }]
    )
    return response.content[0].text.strip()


print("=" * 70)
print("FULL RAG PIPELINE")
print("=" * 70)

for q in test_questions:
    route = routing_results[q]
    chunks = retrieve(q, route["indexes"], k=3)
    answer = generate_answer(q, chunks)

    print(f"\nQ: {q}")
    print(f"   intent  : {route['intent']}")
    print(f"   searched: {route['indexes']}")
    print(f"   chunks retrieved ({len(chunks)}):")
    for c in chunks:
        print(f"     [{c['collection']}] score={c['score']:.3f} — {c['content'][:80].strip()}...")
    print(f"   answer  : {answer[:300]}")
    print("-" * 70)

---
## Section 6 — RAG Evaluation with RAGAS

Build a golden dataset, run the full pipeline, and measure faithfulness, answer relevance, and context recall.

In [ ]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_recall
from datasets import Dataset

# Golden dataset — 5 Q/A pairs with known correct answers from indexed data
golden = [
    {
        "question": "What is my total P&L this month?",
        "ground_truth": "Total P&L for May 2026 is $43,035.50 ($4,820.50 realized, $38,215.00 unrealized).",
        "indexes": ["portfolio"]
    },
    {
        "question": "What is my current NVDA position?",
        "ground_truth": "50 shares of NVDA at avg cost $620.00, current price $875.40, P&L +41.19%.",
        "indexes": ["portfolio"]
    },
    {
        "question": "What did Barclays say about NVDA?",
        "ground_truth": "Barclays reiterated Overweight on NVDA with a $1,050 price target. Blackwell Ultra ramp is ahead of estimates. Key risk: export controls on H20 China sales could cut FY2027 revenue by $4B.",
        "indexes": ["market_news"]
    },
    {
        "question": "Which sector has the most concentration in my portfolio?",
        "ground_truth": "Technology sector dominates with 4 out of 5 positions (AAPL, MSFT, NVDA, GOOGL). Only JPM is in Financials.",
        "indexes": ["portfolio"]
    },
    {
        "question": "What is my account NAV and cash balance?",
        "ground_truth": "Account NAV is $287,450 with $18,200 in cash. No margin is being used.",
        "indexes": ["portfolio"]
    }
]

# Run pipeline for each golden item
eval_records = []
for item in golden:
    chunks = retrieve(item["question"], item["indexes"], k=3)
    answer = generate_answer(item["question"], chunks)
    eval_records.append({
        "question": item["question"],
        "answer": answer,
        "contexts": [c["content"] for c in chunks],
        "ground_truth": item["ground_truth"]
    })

dataset = Dataset.from_list(eval_records)

results = evaluate(
    dataset,
    metrics=[faithfulness, answer_relevancy, context_recall]
)

print("=" * 70)
print("RAGAS EVALUATION RESULTS")
print("=" * 70)
print(f"{'Question':<45} {'Faith':>6} {'Relev':>6} {'Recall':>7}")
print("-" * 68)
for i, row in enumerate(eval_records):
    q = row["question"][:44]
    f_score = results["faithfulness"][i] if hasattr(results["faithfulness"], '__getitem__') else results["faithfulness"]
    r_score = results["answer_relevancy"][i] if hasattr(results["answer_relevancy"], '__getitem__') else results["answer_relevancy"]
    cr_score = results["context_recall"][i] if hasattr(results["context_recall"], '__getitem__') else results["context_recall"]
    print(f"{q:<45} {f_score:>6.3f} {r_score:>6.3f} {cr_score:>7.3f}")

print("\nAggregate scores:")
print(results)

This is what your critic agent should be measuring. **Faithfulness** alone is not enough — context recall catches the case where retrieval failed upstream and Claude faithfully used the wrong information. A system can score 1.0 on faithfulness (never hallucinating) while scoring 0.2 on context recall (almost always retrieving the wrong chunks). Both metrics are required to trust the pipeline.

---
## Section 7 — Index Freshness Demonstration

No API calls. Simulating the FinMentor staleness bug — what happens when real-time financial data lives in an index.

In [ ]:
from datetime import datetime, timedelta

# ── Simulate the bug ────────────────────────────────────────────────────────

# State at index time T
T = datetime(2026, 5, 19, 9, 30)
nvda_at_T = {"ticker": "NVDA", "shares": 50, "current_price": 875.40, "indexed_at": T.isoformat()}
stale_position_value = nvda_at_T["shares"] * nvda_at_T["current_price"]

# Price update at T+1 (30 minutes later)
T1 = T + timedelta(minutes=30)
nvda_actual_price_at_T1 = 920.40  # NVDA moved up $45
actual_position_value = nvda_at_T["shares"] * nvda_actual_price_at_T1
staleness_error = actual_position_value - stale_position_value

print("=== FinMentor Staleness Bug ===")
print(f"Index snapshot time    : {T.strftime('%H:%M')}")
print(f"Query time             : {T1.strftime('%H:%M')}")
print(f"NVDA price in index    : ${nvda_at_T['current_price']:.2f}")
print(f"NVDA actual price      : ${nvda_actual_price_at_T1:.2f}")
print(f"Position value (stale) : ${stale_position_value:,.2f}  ← what RAG returns")
print(f"Position value (actual): ${actual_position_value:,.2f}  ← what user expects")
print(f"Error                  : ${staleness_error:,.2f} ({staleness_error/stale_position_value*100:.1f}%)")
print()
print("User question: 'What is my NVDA position worth?'")
print(f"RAG answer (wrong): Your NVDA position (50 shares) is worth ${stale_position_value:,.2f} based on the current price of ${nvda_at_T['current_price']}.")
print(f"Correct answer    : Your NVDA position (50 shares) is worth ${actual_position_value:,.2f} at the current price of ${nvda_actual_price_at_T1}.")

# ── The fix ─────────────────────────────────────────────────────────────────
print("\n=== The Fix: Data Classification by Freshness Requirement ===")
data_classification = [
    ("Real-time prices",        "NEVER index — fetch from IBKR at query time",      "< 1 second"),
    ("Account NAV / balances",  "NEVER index — fetch from IBKR at query time",      "< 1 second"),
    ("Analyst reports",         "Index OK — refresh daily",                          "24 hours"),
    ("Historical transactions", "Index OK — append-only, immutable",                 "N/A"),
    ("Account metadata",        "Index OK — refresh weekly",                         "1 week"),
    ("Position quantities",     "Index OK for concentration queries — refresh daily", "24 hours"),
]
print(f"{'Data Type':<30} {'Strategy':<45} {'Max Staleness':>15}")
print("-" * 92)
for dtype, strategy, max_staleness in data_classification:
    print(f"{dtype:<30} {strategy:<45} {max_staleness:>15}")

This is almost certainly the source of FinMentor's accuracy bug. Real-time financial data should **never** be indexed — it should be fetched fresh from IBKR at query time and injected directly into context. Index staleness is a **correctness bug**, not a performance bug. The model cannot tell that its retrieved context is stale — it answers confidently with wrong numbers.

---
## Section 8 — Key Observations

1. **Chunking strategy determines retrieval quality more than embedding model choice.** A great embedding model retrieving split-boundary fixed chunks will underperform a mediocre model retrieving semantically coherent chunks. Get the chunks right first.

2. **Query routing is a cheap Claude call that saves expensive multi-index searches on every query.** One classification call before retrieval prevents the portfolio index from being searched for news questions and vice versa. Routing errors are silent — the model still answers, just with the wrong context.

3. **Faithfulness without context recall is a false sense of security.** Claude can be perfectly faithful to retrieved context (score 1.0) while answering incorrectly because retrieval surfaced the wrong chunks (context recall 0.2). Both metrics are required to trust a RAG pipeline.

4. **Real-time financial data should never be indexed.** Index staleness is a correctness bug. The model answers confidently with stale numbers because it cannot detect that the retrieval timestamp predates a price move. The fix is architectural: bypass the index for price-sensitive queries and inject live IBKR data directly at query time.

In [ ]:
print("Commit message:")
print('  04: RAG systems — chunking, vector index, query routing, RAGAS evaluation, freshness demo')
print()
print("Push command:")
print("  git add 04-rag-systems/ README.md")
print("  git commit -m '04: RAG systems — chunking, vector index, query routing, RAGAS evaluation, freshness demo'")
print("  git push origin main")